In [ ]:
# Cell 1: Title and Google Drive mount
from IPython.display import Markdown, display
display(Markdown('# Frozen VAE PCA Workflow (Varimax-first)'))
display(Markdown(
    'This notebook keeps the frozen VAE and PCA assets fixed, uses Varimax as the default control space, '
    'locks one Intensity axis, and then discovers additional metric-aware control directions.'
))

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


In [ ]:
# Cell 2: Repository, frozen manifest, imports, and output folders
display(Markdown('### Cell 2: Repository, frozen manifest, imports, and output folders'))
from pathlib import Path
import json, os, pickle, shutil, subprocess, sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio

REPO_URL = 'https://github.com/cindy-77jiayi/thesis_hapticAE.git'
PROJECT_ROOT = Path('/content/thesis_hapticAE')
BRANCH = None
FORCE_CLEAN_CLONE = False
INSTALL_REQUIREMENTS = True

if FORCE_CLEAN_CLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not (PROJECT_ROOT / '.git').exists():
    clone_cmd = ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    if BRANCH:
        clone_cmd = ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)]
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all'], check=False)
    if BRANCH:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=False)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=False)

print('Current branch:')
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], check=True)

requirements_path = PROJECT_ROOT / 'requirements.txt'
if INSTALL_REQUIREMENTS and requirements_path.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], check=True)

sys.path.insert(0, str(PROJECT_ROOT))

import importlib
for mod in list(sys.modules):
    if mod == 'src' or mod.startswith('src.'):
        del sys.modules[mod]
importlib.invalidate_caches()

DRIVE_ROOT = Path('/content/drive/MyDrive')
FROZEN_ROOT = DRIVE_ROOT / 'thesis' / 'frozen_model_outputs'
FROZEN_MANIFEST_PATH = FROZEN_ROOT / 'latest_frozen_manifest.json'
if not FROZEN_MANIFEST_PATH.exists():
    FROZEN_MANIFEST_PATH = FROZEN_ROOT / 'vae_balanced' / 'frozen_manifest.json'
if not FROZEN_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Frozen manifest not found: {FROZEN_MANIFEST_PATH}')

FROZEN_MANIFEST = json.loads(FROZEN_MANIFEST_PATH.read_text(encoding='utf-8'))
FROZEN_RUN_DIR = Path(FROZEN_MANIFEST.get('frozen_run_dir', FROZEN_MANIFEST_PATH.parent))
CONFIG_PATH = Path(FROZEN_MANIFEST['config_path'])
CHECKPOINT_PATH = Path(FROZEN_MANIFEST['checkpoint_path'])
FROZEN_PCA_DIR = Path(FROZEN_MANIFEST.get('pca_dir') or (FROZEN_RUN_DIR / 'pca'))
FROZEN_CONTROLS_DIR = Path(FROZEN_MANIFEST.get('controls_dir') or (FROZEN_RUN_DIR / 'controls'))
FROZEN_LATENT_PATH = Path(FROZEN_MANIFEST.get('latent_path') or (FROZEN_PCA_DIR / 'Z_mu.npy'))
SAMPLE_IDS_PATH = Path(FROZEN_MANIFEST.get('sample_ids_path') or (FROZEN_PCA_DIR / 'sample_ids.json'))


def _looks_like_dataset_root(path: Path) -> bool:
    return path.exists() and any((path / sub).exists() for sub in ('expertvoted', 'uservoted'))


def _discover_dataset_root() -> Path | None:
    env_dir = os.environ.get('HAPTIC_DATA_DIR')
    candidates = []
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.extend([
        Path('/content/hapticgen-dataset'),
        DRIVE_ROOT / 'hapticgen-dataset',
        DRIVE_ROOT / 'thesis' / 'hapticgen-dataset',
        PROJECT_ROOT / 'hapticgen-dataset',
    ])
    for candidate in candidates:
        if _looks_like_dataset_root(candidate):
            return candidate
    return None


DATA_DIR = _discover_dataset_root()

ANALYSIS_NAME = 'frozen_vae_pca_workflow'
OUTPUT_DIR = FROZEN_RUN_DIR / 'analysis' / ANALYSIS_NAME
LATENT_DIR = OUTPUT_DIR / 'latent_cache'
PCA_DIR = OUTPUT_DIR / 'pca_cache'
VARIMAX_DIR = OUTPUT_DIR / 'varimax_cache'
SWEEP_DIR = OUTPUT_DIR / 'sweeps'
SUMMARY_DIR = OUTPUT_DIR / 'summary'
FAMILY_DIR = OUTPUT_DIR / 'family_directions'
COMPARISON_DIR = OUTPUT_DIR / 'comparison'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for d in [OUTPUT_DIR, LATENT_DIR, PCA_DIR, VARIMAX_DIR, SWEEP_DIR, SUMMARY_DIR, FAMILY_DIR, COMPARISON_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.pipelines.latent_extraction import extract_latent_vectors
from src.pipelines.control_spec import compute_control_ranges
from src.pipelines.pca_control import (
    control_to_latent,
    fit_pca_pipeline,
    fit_rotated_pca_pipeline,
    get_explained_variance_ratio,
    play_sweep,
    plot_sweep,
    sweep_axis,
    sweep_direction,
)
from src.eval.pc_validation import (
    DEFAULT_CAUTION_METRICS,
    DEFAULT_METRIC_FAMILY_MAP,
    ENGINEERING_CONTROL_FAMILIES,
    build_control_samples_from_sweep_payload,
    build_family_profiles,
    choose_anchor_indices,
    compute_axis_alignment,
    discover_metric_family_directions,
    summarize_candidate_axes,
    summarize_multi_anchor_metrics,
)

print('Frozen manifest:', FROZEN_MANIFEST_PATH)
print('Frozen run dir:', FROZEN_RUN_DIR)
print('CONFIG_PATH:', CONFIG_PATH, 'exists=', CONFIG_PATH.exists())
print('CHECKPOINT_PATH:', CHECKPOINT_PATH, 'exists=', CHECKPOINT_PATH.exists())
print('FROZEN_PCA_DIR:', FROZEN_PCA_DIR, 'exists=', FROZEN_PCA_DIR.exists())
print('FROZEN_LATENT_PATH:', FROZEN_LATENT_PATH, 'exists=', FROZEN_LATENT_PATH.exists())
print('SAMPLE_IDS_PATH:', SAMPLE_IDS_PATH, 'exists=', SAMPLE_IDS_PATH.exists())
print('DATA_DIR:', DATA_DIR if DATA_DIR is not None else 'not found (ok if frozen latent/sample_ids exist)')
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# Cell 3: Load config, sample ids, and VAE checkpoint
display(Markdown('### Cell 3: Load config, sample ids, and VAE checkpoint'))

config = load_config(str(CONFIG_PATH))
set_seed(config.get('seed', 42))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
T = int(config['data']['T'])
sr = int(config['data']['sr'])

data = None
sample_ids = []
if SAMPLE_IDS_PATH.exists():
    sample_ids = json.loads(SAMPLE_IDS_PATH.read_text(encoding='utf-8'))
    print('Loaded frozen sample ids:', len(sample_ids))
else:
    print('Frozen sample_ids.json not found; sample ids will be rebuilt only if dataset access is needed.')

model = build_model(config, DEVICE)
load_checkpoint(model, str(CHECKPOINT_PATH), DEVICE)
model.eval()

print('Device:', DEVICE)
print('Signal length:', T)
print('Sample rate:', sr)


In [ ]:
# Cell 4: Load frozen latent cache or optionally re-extract mu
display(Markdown('### Cell 4: Load frozen latent cache or optionally re-extract mu'))

LATENT_PATH = LATENT_DIR / 'Z_mu.npy'
if LATENT_PATH.exists():
    Z = np.load(LATENT_PATH)
    print('Loaded analysis latent cache:', LATENT_PATH, Z.shape)
elif FROZEN_LATENT_PATH.exists():
    Z = np.load(FROZEN_LATENT_PATH)
    np.save(LATENT_PATH, Z.astype(np.float32))
    print('Copied frozen latent cache into analysis dir:', LATENT_PATH, Z.shape)
else:
    if DATA_DIR is None:
        raise FileNotFoundError(
            'No dataset root was found and no frozen latent cache is available. '
            'Either mount the dataset and set HAPTIC_DATA_DIR, or regenerate the frozen latent cache first.'
        )
    data = build_dataloaders(config, str(DATA_DIR), batch_size=64, full_dataset=True)
    sample_ids = [Path(p).name for p in data['wav_files']]
    Z = extract_latent_vectors(model, data['all_loader'], DEVICE)
    np.save(LATENT_PATH, Z.astype(np.float32))
    print('Extracted and saved latent cache:', LATENT_PATH, Z.shape)

if not sample_ids:
    if SAMPLE_IDS_PATH.exists():
        sample_ids = json.loads(SAMPLE_IDS_PATH.read_text(encoding='utf-8'))
    else:
        sample_ids = [f'sample_{i:05d}' for i in range(Z.shape[0])]
        print('Using synthetic sample ids because sample_ids.json is unavailable.')

assert len(sample_ids) == Z.shape[0], f'sample_ids length {len(sample_ids)} does not match latent rows {Z.shape[0]}'
print('Latent rows / sample ids:', Z.shape[0], len(sample_ids))


In [ ]:
# Cell 5: Load frozen PCA artifacts or optionally refit baseline PCA
display(Markdown('### Cell 5: Load frozen PCA artifacts or optionally refit baseline PCA'))

PCA_PIPE_PATH = FROZEN_PCA_DIR / 'pca_pipe.pkl'
PCA_SCORE_PATH = FROZEN_PCA_DIR / 'Z_pca.npy'

if PCA_PIPE_PATH.exists() and PCA_SCORE_PATH.exists():
    with open(PCA_PIPE_PATH, 'rb') as f:
        baseline_pipe = pickle.load(f)
    Z_pca = np.load(PCA_SCORE_PATH)
    print('Loaded frozen PCA artifacts')
else:
    baseline_pipe, Z_pca = fit_pca_pipeline(Z, n_components=8, save_dir=str(PCA_DIR))
    print('Refit baseline PCA because frozen artifacts were missing')

baseline_evr = get_explained_variance_ratio(baseline_pipe)
print('Baseline PCA shape:', Z_pca.shape)
print('Baseline cumulative EVR:', float(np.sum(baseline_evr)))


In [ ]:
# Cell 6: Fit or load rotated Varimax control space
display(Markdown('### Cell 6: Fit or load rotated Varimax control space'))

VARIMAX_PIPE_PATH = VARIMAX_DIR / 'varimax_pipe.pkl'
VARIMAX_SCORE_PATH = VARIMAX_DIR / 'Z_varimax.npy'

if VARIMAX_PIPE_PATH.exists() and VARIMAX_SCORE_PATH.exists():
    with open(VARIMAX_PIPE_PATH, 'rb') as f:
        varimax_pipe = pickle.load(f)
    Z_varimax = np.load(VARIMAX_SCORE_PATH)
    print('Loaded cached Varimax artifacts')
else:
    varimax_pipe, Z_varimax = fit_rotated_pca_pipeline(
        baseline_pipe,
        Z_pca=Z_pca,
        save_dir=str(VARIMAX_DIR),
        prefix='varimax',
    )
    print('Fitted and saved Varimax artifacts')

varimax_evr = get_explained_variance_ratio(varimax_pipe)
print('Varimax shape:', Z_varimax.shape)
print('Varimax cumulative EVR:', float(np.sum(varimax_evr)))

pd.DataFrame({
    'axis': [f'PC{i+1}' for i in range(len(varimax_evr))],
    'explained_variance_ratio': np.round(varimax_evr, 6),
})


In [ ]:
# Cell 7: Varimax sweep setup, engineering taxonomy, and shared helpers
display(Markdown('### Cell 7: Varimax sweep setup, engineering taxonomy, and shared helpers'))

ENGINEERING_FAMILIES = list(ENGINEERING_CONTROL_FAMILIES)
METRIC_FAMILY_MAP = dict(DEFAULT_METRIC_FAMILY_MAP)
CAUTION_LABEL_METRICS = set(DEFAULT_CAUTION_METRICS)

EXCLUSIVE_INTENSITY_FAMILY = 'Amplitude / Intensity (Voltage)'
INTENSITY_AXIS_CANDIDATES = ['PC1', 'PC2']

N_ANCHORS = 5
N_SWEEP_STEPS = 21
SINGLE_AXIS_STEPS = 9
VARIMAX_LISTEN_AXES = [0, 1, 3, 5]

anchor_indices = choose_anchor_indices(Z_varimax, n_anchors=N_ANCHORS, seed=config.get('seed', 42))
anchor_table = pd.DataFrame([
    {'anchor_id': i, 'row_index': int(idx), 'sample_id': sample_ids[int(idx)]}
    for i, idx in enumerate(anchor_indices)
])
varimax_anchor_refs = Z_varimax[anchor_indices].astype(np.float32)
baseline_anchor_refs = Z_pca[anchor_indices].astype(np.float32)

varimax_ranges = compute_control_ranges(Z_varimax)
baseline_ranges = compute_control_ranges(Z_pca)

family_taxonomy_table = pd.DataFrame([
    {'control_family': family, 'metrics': ', '.join(sorted([m for m, fam in METRIC_FAMILY_MAP.items() if fam == family]))}
    for family in ENGINEERING_FAMILIES
])

def _audio_norm(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return np.clip(sig / (np.max(np.abs(sig)) + 1e-8), -1.0, 1.0)

def _axis_range(axis, ranges):
    match = [r for r in ranges if int(r['axis']) == int(axis)]
    if not match:
        raise ValueError(f'No sweep range found for axis {axis}')
    return (float(match[0]['p5']), float(match[0]['p95']))

def _direction_range(score_matrix, direction, percentiles=(5.0, 95.0)):
    direction = np.asarray(direction, dtype=np.float32).reshape(-1)
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    projection = np.asarray(score_matrix, dtype=np.float32) @ direction
    lo, hi = np.percentile(projection, percentiles)
    return (float(lo), float(hi))

def run_multi_anchor_axis_sweeps(pipe, anchor_refs, ranges, sweep_name, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    payload_path = out_dir / f'{sweep_name}_multi_anchor_sweeps.json'

    if payload_path.exists():
        payload = json.loads(payload_path.read_text(encoding='utf-8'))
        print('Loaded cached sweep payload:', payload_path)
        return payload

    payload = {
        'method': sweep_name,
        'n_steps': int(N_SWEEP_STEPS),
        'sweeps': {},
    }

    for axis in range(anchor_refs.shape[1]):
        pc = f'PC{axis + 1}'
        lo, hi = _axis_range(axis, ranges)
        payload['sweeps'][pc] = []
        for anchor_id, reference in enumerate(anchor_refs):
            sweep = sweep_axis(
                pipe,
                model,
                DEVICE,
                axis=axis,
                sweep_range=(lo, hi),
                n_steps=N_SWEEP_STEPS,
                T=T,
                sr=sr,
                reference=reference,
                with_metrics=True,
            )
            payload['sweeps'][pc].append({
                'anchor_id': int(anchor_id),
                'row_index': int(anchor_indices[anchor_id]),
                'sample_id': sample_ids[int(anchor_indices[anchor_id])],
                'values': [float(v) for v in sweep['values']],
                'metrics': [{k: float(v) for k, v in item.items()} for item in sweep['metrics']],
            })

    payload_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Saved sweep payload:', payload_path)
    return payload

def summarize_sweep_payload(sweep_payload):
    return {
        pc: summarize_multi_anchor_metrics(items)
        for pc, items in sweep_payload['sweeps'].items()
    }

def build_method_overview(method_name, summary_payload):
    df = pd.DataFrame(summary_payload['rows'])
    return {
        'method': method_name,
        'n_use': int((df['use_recommendation'] == 'use').sum()),
        'n_review': int((df['use_recommendation'] == 'review').sum()),
        'mean_overall_score': round(float(df['overall_score'].mean()), 6),
        'mean_selectivity': round(float(df['selectivity'].mean()), 6),
        'mean_family_selectivity': round(float(df['family_selectivity'].mean()), 6),
        'mean_cross_anchor_consistency': round(float(df['cross_anchor_consistency'].mean()), 6),
    }

display(anchor_table)
display(family_taxonomy_table)


In [ ]:
# Cell 8: Varimax multi-anchor sweeps
display(Markdown('### Cell 8: Varimax multi-anchor sweeps'))

VARIMAX_SWEEP_SUBDIR = SWEEP_DIR / 'varimax'
varimax_sweep_payload = run_multi_anchor_axis_sweeps(
    varimax_pipe,
    varimax_anchor_refs,
    varimax_ranges,
    sweep_name='varimax',
    out_dir=VARIMAX_SWEEP_SUBDIR,
)
varimax_pc_summaries = summarize_sweep_payload(varimax_sweep_payload)

pd.DataFrame(varimax_ranges)


In [ ]:
# Cell 9: Varimax single-axis listening and visualization
display(Markdown('### Cell 9: Varimax single-axis listening and visualization'))

VARIMAX_SINGLE_AXIS_DIR = OUTPUT_DIR / 'varimax_single_axis_sweeps'
VARIMAX_SINGLE_AXIS_DIR.mkdir(parents=True, exist_ok=True)
LISTEN_ANCHOR_ID = 0
listen_reference = varimax_anchor_refs[LISTEN_ANCHOR_ID]

print('Listening reference anchor:', int(anchor_indices[LISTEN_ANCHOR_ID]), sample_ids[int(anchor_indices[LISTEN_ANCHOR_ID])])

for axis in VARIMAX_LISTEN_AXES:
    pc_name = f'PC{axis + 1}'
    sweep_range = _axis_range(axis, varimax_ranges)
    print(f"\n{'=' * 60}")
    print(f'{pc_name} sweep range: {sweep_range[0]:+.3f} to {sweep_range[1]:+.3f}')
    print(f"{'=' * 60}")

    result = sweep_axis(
        varimax_pipe,
        model,
        DEVICE,
        axis=axis,
        sweep_range=sweep_range,
        n_steps=SINGLE_AXIS_STEPS,
        T=T,
        sr=sr,
        reference=listen_reference,
        with_metrics=True,
    )
    plot_sweep(result, sr=sr, save_path=str(VARIMAX_SINGLE_AXIS_DIR / f'{pc_name}_waveforms.png'))
    play_sweep(result, sr=sr)
    np.savez(
        VARIMAX_SINGLE_AXIS_DIR / f'{pc_name}_single_axis_sweep.npz',
        values=np.asarray(result['values'], dtype=np.float32),
        signals=np.asarray(result['signals'], dtype=np.float32),
    )

print('Saved Varimax listening assets to:', VARIMAX_SINGLE_AXIS_DIR)


In [ ]:
# Cell 10: Raw Varimax family profiles before forced PC locks
display(Markdown('### Cell 10: Raw Varimax family profiles before forced PC locks'))

display(Markdown('''**Touch Check Note**

- `PC2 / Intensity`: should mainly feel like stronger or weaker overall vibration, with the same general character.
- `PC1 / Frequency`: should mainly feel like sharper/finer/buzzier versus thicker/heavier, not just stronger.'''))

FORCED_PC_FAMILY_LOCKS = {
    'PC1': 'Frequency / Spectral Balance',
    'PC2': 'Amplitude / Intensity (Voltage)',
}
LOCKED_PC_ORDER = ['PC1', 'PC2']
LOCKED_FAMILIES = set(FORCED_PC_FAMILY_LOCKS.values())

varimax_candidate_summary = summarize_candidate_axes(
    varimax_pc_summaries,
    explained_variance_ratio=varimax_evr,
    metric_family_map=METRIC_FAMILY_MAP,
    caution_metrics=CAUTION_LABEL_METRICS,
)

varimax_candidate_df = pd.DataFrame(varimax_candidate_summary['rows'])
varimax_candidate_df['recommendation_rank'] = varimax_candidate_df['use_recommendation'].map({'use': 2, 'review': 1, 'disable': 0})
varimax_candidate_df = varimax_candidate_df.sort_values(
    ['recommendation_rank', 'overall_score'],
    ascending=[False, False],
).reset_index(drop=True)

family_profile_rows = []
for pc_name, family_profiles in varimax_candidate_summary['family_profiles'].items():
    for family, info in family_profiles.items():
        family_profile_rows.append({
            'pc': pc_name,
            'control_family': family,
            'family_score': info['score'],
            'top_metric': info['top_metric'],
            'representative_metrics': ', '.join(info['representative_metrics']),
        })
family_profile_df = pd.DataFrame(family_profile_rows).sort_values(['pc', 'family_score'], ascending=[True, False])

forced_lock_rows = []
for pc_name in LOCKED_PC_ORDER:
    family_name = FORCED_PC_FAMILY_LOCKS[pc_name]
    match = family_profile_df[
        (family_profile_df['pc'] == pc_name) &
        (family_profile_df['control_family'] == family_name)
    ].copy()
    if match.empty:
        raise RuntimeError(f'Could not find family profile for {pc_name} -> {family_name}')
    merged = match.merge(
        varimax_candidate_df[['pc', 'overall_score', 'confidence', 'use_recommendation']],
        on='pc',
        how='left',
    )
    forced_lock_rows.append(merged.iloc[0].to_dict())
forced_lock_df = pd.DataFrame(forced_lock_rows)

summary_csv = SUMMARY_DIR / 'varimax_candidate_summary.csv'
summary_json = SUMMARY_DIR / 'varimax_candidate_summary.json'
family_profile_csv = SUMMARY_DIR / 'varimax_family_profiles.csv'
forced_lock_csv = SUMMARY_DIR / 'forced_pc_family_locks.csv'

varimax_candidate_df.to_csv(summary_csv, index=False)
summary_json.write_text(json.dumps(varimax_candidate_summary, indent=2), encoding='utf-8')
family_profile_df.to_csv(family_profile_csv, index=False)
forced_lock_df.to_csv(forced_lock_csv, index=False)

pc12_profile_df = family_profile_df[family_profile_df['pc'].isin(LOCKED_PC_ORDER)].copy()
print('Forced PC locks for the next steps:')
display(forced_lock_df)
print('PC1 / PC2 supporting family profiles:')
display(pc12_profile_df)


In [ ]:
# Cell 11: Metric-aware family direction discovery for unlocked families only
display(Markdown('### Cell 11: Metric-aware family direction discovery for unlocked families only'))

FAMILY_DIRECTION_DIR = FAMILY_DIR / 'varimax_metric_aware'
FAMILY_DIRECTION_DIR.mkdir(parents=True, exist_ok=True)

control_samples, metric_samples, sample_metadata = build_control_samples_from_sweep_payload(
    varimax_sweep_payload,
    varimax_anchor_refs,
)

locked_family_directions = {
    FORCED_PC_FAMILY_LOCKS['PC1']: np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32),
    FORCED_PC_FAMILY_LOCKS['PC2']: np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32),
}
locked_pc_controls = {
    'PC1': {'pc': 'PC1', 'axis': 0, 'control_family': FORCED_PC_FAMILY_LOCKS['PC1'], 'mode': 'locked_pc'},
    'PC2': {'pc': 'PC2', 'axis': 1, 'control_family': FORCED_PC_FAMILY_LOCKS['PC2'], 'mode': 'locked_pc'},
}
intensity_reference = locked_pc_controls['PC2']
frequency_reference = locked_pc_controls['PC1']

target_families = [
    family for family in ENGINEERING_FAMILIES
    if family not in LOCKED_FAMILIES
]

discovered_family_directions = discover_metric_family_directions(
    control_samples=control_samples,
    metric_samples=metric_samples,
    metric_family_map=METRIC_FAMILY_MAP,
    target_families=target_families,
    locked_directions=locked_family_directions,
    ridge_alpha=1.0,
)

def classify_family_direction(profile, family_selectivity):
    rho = float(profile['mean_abs_rho'])
    effect = float(profile['mean_effect'])
    consistency = float(profile['sign_consistency'])
    if rho >= 0.75 and effect >= 0.18 and consistency >= 0.80 and family_selectivity >= 1.40:
        return 'strong', 'use'
    if rho >= 0.55 and effect >= 0.10 and consistency >= 0.65 and family_selectivity >= 1.10:
        return 'moderate', 'review'
    return 'weak', 'disable'

family_direction_payload = {
    'locked_pc_controls': locked_pc_controls,
    'directions': {},
}
family_direction_rows = []

for family, info in discovered_family_directions['directions'].items():
    direction = np.asarray(info['direction'], dtype=np.float32)
    sweep_range = _direction_range(Z_varimax, direction)
    sweep_items = []

    for anchor_id, reference in enumerate(varimax_anchor_refs):
        sweep = sweep_direction(
            varimax_pipe,
            model,
            DEVICE,
            direction=direction,
            sweep_range=sweep_range,
            n_steps=N_SWEEP_STEPS,
            T=T,
            sr=sr,
            reference=reference,
            with_metrics=True,
            name=family,
        )
        sweep_items.append({
            'anchor_id': int(anchor_id),
            'row_index': int(anchor_indices[anchor_id]),
            'sample_id': sample_ids[int(anchor_indices[anchor_id])],
            'values': [float(v) for v in sweep['values']],
            'metrics': [{k: float(v) for k, v in metric_row.items()} for metric_row in sweep['metrics']],
        })

    summary = summarize_multi_anchor_metrics(sweep_items)
    family_profiles = build_family_profiles(
        summary['per_metric'],
        metric_family_map=METRIC_FAMILY_MAP,
        caution_metrics=CAUTION_LABEL_METRICS,
    )
    profile = family_profiles.get(family)
    if profile is None:
        continue

    off_family_scores = np.array(
        [p['score'] for other_family, p in family_profiles.items() if other_family != family and p['score'] > 0.0],
        dtype=float,
    )
    family_selectivity = float(profile['score'] / (np.mean(off_family_scores) + 1e-8)) if off_family_scores.size else float(profile['score'])
    confidence, use_recommendation = classify_family_direction(profile, family_selectivity)
    dominant_metrics = '; '.join(
        [
            f"{metric_name}:{metric_info['mean_rho']:+.2f}/eff{metric_info['mean_effect']:.2f}/cons{metric_info['sign_consistency']:.2f}"
            for metric_name, metric_info in summary['ranking'][:4]
        ]
    )

    family_direction_rows.append({
        'control_family': family,
        'source_metrics': ', '.join(info['source_metrics']),
        'top_metric': profile['top_metric'],
        'dominant_metrics': dominant_metrics,
        'target_correlation': float(info['target_correlation']),
        'family_score': round(float(profile['score']), 6),
        'monotonicity_abs_rho': round(float(profile['mean_abs_rho']), 4),
        'effect_size_relative': round(float(profile['mean_effect']), 4),
        'cross_anchor_consistency': round(float(profile['sign_consistency']), 4),
        'family_selectivity': round(float(family_selectivity), 4),
        'confidence': confidence,
        'use_recommendation': use_recommendation,
    })

    family_direction_payload['directions'][family] = {
        'direction': direction.tolist(),
        'range': [float(sweep_range[0]), float(sweep_range[1])],
        'source_metrics': list(info['source_metrics']),
        'target_correlation': float(info['target_correlation']),
        'sweeps': sweep_items,
    }

family_direction_df = pd.DataFrame(family_direction_rows).sort_values(
    ['use_recommendation', 'family_score'],
    ascending=[True, False],
).reset_index(drop=True)

family_summary_csv = FAMILY_DIRECTION_DIR / 'family_direction_summary.csv'
family_summary_json = FAMILY_DIRECTION_DIR / 'family_direction_summary.json'
family_payload_json = FAMILY_DIRECTION_DIR / 'family_direction_sweeps.json'

family_direction_df.to_csv(family_summary_csv, index=False)
family_summary_json.write_text(json.dumps(family_direction_rows, indent=2), encoding='utf-8')
family_payload_json.write_text(json.dumps(family_direction_payload, indent=2), encoding='utf-8')

display(family_direction_df)


In [ ]:
# Cell 12: Lock PC1 and PC2, then relabel remaining PCs
display(Markdown('### Cell 12: Lock PC1 and PC2, then relabel remaining PCs'))

LOCKED_CONTROL_DIR = SUMMARY_DIR / 'locked_controls'
LOCKED_CONTROL_DIR.mkdir(parents=True, exist_ok=True)

locked_pc_controls_table = pd.DataFrame([
    {'pc': 'PC1', 'control_family': FORCED_PC_FAMILY_LOCKS['PC1'], 'mode': 'locked_pc', 'top_metric': 'low_high_band_ratio'},
    {'pc': 'PC2', 'control_family': FORCED_PC_FAMILY_LOCKS['PC2'], 'mode': 'locked_pc', 'top_metric': 'envelope_area'},
])

remaining_pc_rows = []
for pc_name in [f'PC{i}' for i in range(3, 9)]:
    family_rows = family_profile_df[
        (family_profile_df['pc'] == pc_name) &
        (~family_profile_df['control_family'].isin(LOCKED_FAMILIES))
    ].sort_values('family_score', ascending=False)
    if family_rows.empty:
        continue
    chosen = family_rows.iloc[0]
    raw_row = varimax_candidate_df[varimax_candidate_df['pc'] == pc_name].iloc[0]
    remaining_pc_rows.append({
        'pc': pc_name,
        'final_pc_label': chosen['control_family'],
        'top_metric': chosen['top_metric'],
        'family_score': chosen['family_score'],
        'raw_overall_score': raw_row['overall_score'],
        'confidence': raw_row['confidence'],
        'use_recommendation': raw_row['use_recommendation'],
        'locked_frequency_elsewhere': True,
        'locked_intensity_elsewhere': True,
    })
remaining_pc_label_df = pd.DataFrame(remaining_pc_rows).sort_values(
    ['raw_overall_score', 'family_score'],
    ascending=[False, False],
).reset_index(drop=True)

final_control_registry_rows = [
    {
        'control_id': 'PC1',
        'control_family': FORCED_PC_FAMILY_LOCKS['PC1'],
        'mode': 'locked_pc',
        'top_metric': 'low_high_band_ratio',
        'overall_score': None,
        'use_recommendation': 'use',
    },
    {
        'control_id': 'PC2',
        'control_family': FORCED_PC_FAMILY_LOCKS['PC2'],
        'mode': 'locked_pc',
        'top_metric': 'envelope_area',
        'overall_score': None,
        'use_recommendation': 'use',
    },
]
final_control_registry_rows.extend([
    {
        'control_id': row['pc'],
        'control_family': row['final_pc_label'],
        'mode': 'varimax_pc',
        'top_metric': row['top_metric'],
        'overall_score': row['raw_overall_score'],
        'use_recommendation': row['use_recommendation'],
    }
    for _, row in remaining_pc_label_df.iterrows()
])
final_control_registry_df = pd.DataFrame(final_control_registry_rows)

(LOCKED_CONTROL_DIR / 'locked_pc_controls.json').write_text(
    json.dumps(locked_pc_controls, indent=2),
    encoding='utf-8',
)
remaining_pc_label_df.to_csv(LOCKED_CONTROL_DIR / 'remaining_pc_label_summary.csv', index=False)
final_control_registry_df.to_csv(LOCKED_CONTROL_DIR / 'final_control_registry.csv', index=False)

print('Locked PC controls:')
display(locked_pc_controls_table)
print('Remaining PC labeling after excluding Frequency and Intensity:')
display(remaining_pc_label_df[[
    'pc',
    'final_pc_label',
    'top_metric',
    'raw_overall_score',
    'confidence',
    'use_recommendation',
]])


In [ ]:
# Cell 13: Focused sweeps with PC2 locked
display(Markdown('### Cell 13: Focused sweeps with PC2 locked'))

FOCUSED_SWEEP_DIR = OUTPUT_DIR / 'focused_sweeps'
FOCUSED_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
FOCUS_ANCHOR_ID = 0
focus_reference = varimax_anchor_refs[FOCUS_ANCHOR_ID].copy()

if 'run_focus_axis_sweeps' not in globals():
    def run_focus_axis_sweeps(stage_name, selected_df, reference_control, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        payload = {
            'stage_name': stage_name,
            'anchor_id': int(FOCUS_ANCHOR_ID),
            'row_index': int(anchor_indices[FOCUS_ANCHOR_ID]),
            'sample_id': sample_ids[int(anchor_indices[FOCUS_ANCHOR_ID])],
            'selected_pcs': selected_df.to_dict(orient='records'),
            'sweeps': {},
        }
        for _, row in selected_df.iterrows():
            pc_name = row['pc']
            axis = int(pc_name.replace('PC', '')) - 1
            sweep_range = _axis_range(axis, varimax_ranges)
            result = sweep_axis(
                varimax_pipe,
                model,
                DEVICE,
                axis=axis,
                sweep_range=sweep_range,
                n_steps=SINGLE_AXIS_STEPS,
                T=T,
                sr=sr,
                reference=reference_control,
                with_metrics=True,
            )
            plot_sweep(result, sr=sr, save_path=str(out_dir / f'{pc_name}_waveforms.png'))
            payload['sweeps'][pc_name] = {
                'axis': axis,
                'label': row['label'],
                'top_metric': row['top_metric'],
                'score': float(row['score']),
                'values': [float(v) for v in result['values']],
                'metrics': [{k: float(v) for k, v in metric_row.items()} for metric_row in result['metrics']],
            }
        (out_dir / f'{stage_name}.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
        selected_df.to_csv(out_dir / f'{stage_name}_summary.csv', index=False)
        return payload

pc2_locked_focus_df = varimax_candidate_df[
    (varimax_candidate_df['pc'] != 'PC2') &
    (varimax_candidate_df['control_family'] != FORCED_PC_FAMILY_LOCKS['PC2'])
].copy().sort_values('overall_score', ascending=False).head(3)
pc2_locked_focus_df = pc2_locked_focus_df.assign(
    label=pc2_locked_focus_df['control_family'],
    score=pc2_locked_focus_df['overall_score'],
)
pc2_locked_focus_df = pc2_locked_focus_df[['pc', 'label', 'top_metric', 'score', 'confidence', 'use_recommendation']].reset_index(drop=True)

locked_pc2_payload = run_focus_axis_sweeps(
    'locked_pc2_focus_sweeps',
    pc2_locked_focus_df,
    focus_reference,
    FOCUSED_SWEEP_DIR / 'locked_pc2',
)

display(pc2_locked_focus_df)


In [ ]:
# Cell 14: Focused sweeps with PC1 and PC2 locked
display(Markdown('### Cell 14: Focused sweeps with PC1 and PC2 locked'))

focus_reference_pc12 = varimax_anchor_refs[FOCUS_ANCHOR_ID].copy()
pc12_locked_focus_df = remaining_pc_label_df.sort_values('raw_overall_score', ascending=False).head(3).copy()
pc12_locked_focus_df = pc12_locked_focus_df.assign(
    label=pc12_locked_focus_df['final_pc_label'],
    score=pc12_locked_focus_df['raw_overall_score'],
)
pc12_locked_focus_df = pc12_locked_focus_df[['pc', 'label', 'top_metric', 'score', 'confidence', 'use_recommendation']].reset_index(drop=True)

locked_pc12_payload = run_focus_axis_sweeps(
    'locked_pc1_pc2_focus_sweeps',
    pc12_locked_focus_df,
    focus_reference_pc12,
    FOCUSED_SWEEP_DIR / 'locked_pc1_pc2',
)

display(pc12_locked_focus_df)


In [ ]:
# Cell 15: Output manifest
display(Markdown('### Cell 15: Output manifest'))

output_manifest = {
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'latent_cache': str(LATENT_PATH),
    'varimax_dir': str(VARIMAX_DIR),
    'varimax_sweep_payload': str(VARIMAX_SWEEP_SUBDIR / 'varimax_multi_anchor_sweeps.json'),
    'varimax_candidate_summary_csv': str(SUMMARY_DIR / 'varimax_candidate_summary.csv'),
    'varimax_candidate_summary_json': str(SUMMARY_DIR / 'varimax_candidate_summary.json'),
    'varimax_family_profiles_csv': str(SUMMARY_DIR / 'varimax_family_profiles.csv'),
    'forced_pc_family_locks_csv': str(SUMMARY_DIR / 'forced_pc_family_locks.csv'),
    'locked_pc_controls_json': str(SUMMARY_DIR / 'locked_controls' / 'locked_pc_controls.json'),
    'remaining_pc_label_summary_csv': str(SUMMARY_DIR / 'locked_controls' / 'remaining_pc_label_summary.csv'),
    'final_control_registry_csv': str(SUMMARY_DIR / 'locked_controls' / 'final_control_registry.csv'),
    'family_direction_summary_csv': str(FAMILY_DIR / 'varimax_metric_aware' / 'family_direction_summary.csv'),
    'family_direction_summary_json': str(FAMILY_DIR / 'varimax_metric_aware' / 'family_direction_summary.json'),
    'family_direction_sweeps_json': str(FAMILY_DIR / 'varimax_metric_aware' / 'family_direction_sweeps.json'),
    'locked_pc2_focus_sweeps_json': str(FOCUSED_SWEEP_DIR / 'locked_pc2' / 'locked_pc2_focus_sweeps.json'),
    'locked_pc2_focus_summary_csv': str(FOCUSED_SWEEP_DIR / 'locked_pc2' / 'locked_pc2_focus_sweeps_summary.csv'),
    'locked_pc1_pc2_focus_sweeps_json': str(FOCUSED_SWEEP_DIR / 'locked_pc1_pc2' / 'locked_pc1_pc2_focus_sweeps.json'),
    'locked_pc1_pc2_focus_summary_csv': str(FOCUSED_SWEEP_DIR / 'locked_pc1_pc2' / 'locked_pc1_pc2_focus_sweeps_summary.csv'),
    'figures_dir': str(FIGURE_DIR),
}
output_manifest_path = OUTPUT_DIR / 'varimax_first_output_manifest.json'
output_manifest_path.write_text(json.dumps(output_manifest, indent=2), encoding='utf-8')
print('Saved output manifest:', output_manifest_path)
display(pd.DataFrame([output_manifest]))
